## Libarays

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import seaborn as sns
import os
import tensorflow as tf
import multiprocessing
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical,plot_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Concatenate, BatchNormalization, Dropout, Activation, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split

## Set up envioment

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

print("TensorFlow devices:", tf.config.list_physical_devices())
print("Is GPU available:", tf.config.list_physical_devices('GPU'))

num_cores = multiprocessing.cpu_count()
print(f"Number of CPU cores: {num_cores}")

tf.config.threading.set_intra_op_parallelism_threads(num_cores)
tf.config.threading.set_inter_op_parallelism_threads(num_cores)

## 3. Load and visualize CIFAR-10 dataset

In [ ]:
# Load data
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# CIFAR-10 has 10 classes
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

def visualize_cifar10_samples(X, y, class_names, samples_per_class,random_seed = 2025):
    """
    Displays a grid of sample images for each class in CIFAR-10.
    X: image array, shape (N, 32, 32, 3)
    y: labels array, shape (N, 1)
    class_names: list of 10 class names
    samples_per_class: how many examples to show per class
    """
    np.random.seed(random_seed)
    y = y.squeeze()  # Convert from shape (N,1) to (N,) if needed
    num_classes = len(class_names)
    plt.figure(figsize=(samples_per_class * 2, num_classes * 2))
    for cls_idx in range(num_classes):
        idxs = np.flatnonzero(y == cls_idx)
        idxs = np.random.choice(idxs, samples_per_class, replace=False)
        for i, idx in enumerate(idxs):
            plt.subplot(num_classes, samples_per_class, cls_idx * samples_per_class + i + 1)
            plt.imshow(X[idx])
            plt.axis('off')
            if i == 0:
                plt.title(class_names[cls_idx])
    plt.tight_layout()
    plt.show()

visualize_cifar10_samples(X_train, y_train, class_names,5)

In [ ]:
X_train.shape

## 4. Data preprocessing

In [ ]:
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

In [ ]:
num_classes = 10
y_train_one_hot = to_categorical(y_train, num_classes)
y_test_one_hot = to_categorical(y_test, num_classes)

In [ ]:
X_train, X_val, y_train_one_hot, y_val_one_hot = train_test_split(
    X_train, y_train_one_hot, test_size=0.2, random_state=2025
)

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=15,     # Random rotation degree range
    width_shift_range=0.1, # Random horizontal shift range
    height_shift_range=0.1, # Random vertical shift range
    horizontal_flip=True,  # Random horizontal flips
    zoom_range=0.1,        # Random zoom range
    fill_mode='nearest'    # Fill mode for pixels outside boundaries
)


In [ ]:
print("Training set shape:", X_train.shape)
print("Validation set shape:", X_val.shape)
print("Test set shape:", X_test.shape)
print("Training labels shape:", y_train_one_hot.shape)
print("Validation labels shape:", y_val_one_hot.shape)
print("Test labels shape:", y_test_one_hot.shape)

In [ ]:
def visualize_augmentation(X, y, datagen, class_names, samples=3):
    """Visualize the effects of data augmentation"""
    import matplotlib.pyplot as plt
    
    sample_idx = np.random.randint(0, X.shape[0])
    sample_image = X[sample_idx:sample_idx+1]
    sample_label = np.argmax(y[sample_idx])
    
    plt.figure(figsize=(12, 4))
    plt.subplot(1, samples+1, 1)
    plt.imshow(sample_image[0])
    plt.title(f"Original: {class_names[sample_label]}")
    plt.axis('off')
    
    i = 0
    for batch in datagen.flow(sample_image, batch_size=1):
        i += 1
        plt.subplot(1, samples+1, i+1)
        plt.imshow(batch[0])
        plt.title(f"Augmented #{i}")
        plt.axis('off')
        if i >= samples:
            break
    plt.tight_layout()
    plt.show()

In [ ]:
visualize_augmentation(X_train, y_train_one_hot, datagen, class_names)

In [ ]:
def expand_dataset(X, y, datagen, augmentation_factor=2):
    """
    Expands the dataset by creating augmented versions of each image.
    
    Parameters:
    X: Original images, shape (n_samples, height, width, channels)
    y: Original labels
    datagen: ImageDataGenerator configured with augmentation parameters
    augmentation_factor: How many augmented versions to create for each original image
    
    Returns:
    X_expanded: Expanded set of images including originals and augmented versions
    y_expanded: Corresponding labels for the expanded dataset
    """
    n_samples = X.shape[0]

    X_expanded = np.zeros((n_samples * (augmentation_factor + 1), *X.shape[1:]), dtype=X.dtype)

    if len(y.shape) > 1:
        y_expanded = np.zeros((n_samples * (augmentation_factor + 1), y.shape[1]), dtype=y.dtype)
    else:
        y_expanded = np.zeros((n_samples * (augmentation_factor + 1),), dtype=y.dtype)

    X_expanded[:n_samples] = X
    y_expanded[:n_samples] = y
    
    print(f"Generating augmented data... this may take some time")
    
    for i in range(n_samples):
        img = X[i:i+1]
        lbl = y[i:i+1]
        
        aug_counter = 0
        for batch in datagen.flow(img, batch_size=1):
            idx = n_samples + i + aug_counter * n_samples
            X_expanded[idx] = batch[0]
            y_expanded[idx] = lbl
            
            aug_counter += 1
            if aug_counter >= augmentation_factor:
                break

        if (i + 1) % 1000 == 0:
            print(f"Processed {i + 1}/{n_samples} images")
    
    print(f"Expanded dataset generation complete!")
    return X_expanded, y_expanded

In [ ]:
augmentation_factor = 3
X_train_expanded, y_train_one_hot_expanded = expand_dataset(
    X_train, y_train_one_hot, datagen, augmentation_factor
)

print(f"Original training set shape: {X_train.shape}")
print(f"Expanded training set shape: {X_train_expanded.shape}")


##  5. Create efficient data pipeline

In [ ]:
def create_dataset_pipeline(X, y, batch_size=64, is_training=True):
    """
    Creates a TensorFlow dataset pipeline without on-the-fly augmentation.
    """

    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    if is_training:
        dataset = dataset.shuffle(buffer_size=1000)

    def preprocess(image, label):
        return image, label
    

    dataset = dataset.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

In [ ]:
train_dataset = create_dataset_pipeline(X_train, y_train_one_hot, is_training=True)
val_dataset = create_dataset_pipeline(X_val, y_val_one_hot, is_training=False)
test_dataset = create_dataset_pipeline(X_test, y_test_one_hot, is_training=False)

In [ ]:
train_dataset

##  6. Define the dual-stream CNN model architecture

In [ ]:
def build_dual_stream_model(input_shape=(32, 32, 3), num_classes=10):
    """
    Build a dual-stream CNN model that processes luminance (Y) 
    and chrominance (UV) channels separately before merging
    """
    inputs = Input(shape=input_shape)
    
    y_channel = Lambda(lambda x: tf.image.rgb_to_yuv(x)[:,:,:,0:1])(inputs)
    uv_channels = Lambda(lambda x: tf.image.rgb_to_yuv(x)[:,:,:,1:3])(inputs)
    

    y = Conv2D(32, (3, 3), padding='same')(y_channel)
    y = BatchNormalization()(y)
    y = Activation('relu')(y)
    y = Conv2D(32, (3, 3), padding='same')(y)
    y = BatchNormalization()(y)
    y = Activation('relu')(y)
    y = MaxPooling2D((2, 2))(y)
    y = Dropout(0.2)(y)
    
    y = Conv2D(64, (3, 3), padding='same')(y)
    y = BatchNormalization()(y)
    y = Activation('relu')(y)
    y = Conv2D(64, (3, 3), padding='same')(y)
    y = BatchNormalization()(y)
    y = Activation('relu')(y)
    y = MaxPooling2D((2, 2))(y)
    y = Dropout(0.3)(y)

    uv = Conv2D(32, (3, 3), padding='same')(uv_channels)
    uv = BatchNormalization()(uv)
    uv = Activation('relu')(uv)
    uv = Conv2D(32, (3, 3), padding='same')(uv)
    uv = BatchNormalization()(uv)
    uv = Activation('relu')(uv)
    uv = MaxPooling2D((2, 2))(uv)
    uv = Dropout(0.2)(uv)
    
    uv = Conv2D(64, (3, 3), padding='same')(uv)
    uv = BatchNormalization()(uv)
    uv = Activation('relu')(uv)
    uv = Conv2D(64, (3, 3), padding='same')(uv)
    uv = BatchNormalization()(uv)
    uv = Activation('relu')(uv)
    uv = MaxPooling2D((2, 2))(uv)
    uv = Dropout(0.3)(uv)
    
    merge = Concatenate()([y, uv])
    
 
    x = Conv2D(128, (3, 3), padding='same')(merge)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(128, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.4)(x)

    x = Flatten()(x)
    x = Dense(256)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(num_classes, activation='softmax')(x)
    

    model = Model(inputs=inputs, outputs=x)
    
    return model



In [ ]:
model = build_dual_stream_model()
model.summary()

##  7. Set up model callbacks

In [ ]:
callbacks = [
    ModelCheckpoint(
        'best_model.keras', 
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,             #
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,             
        patience=5,              
        min_lr=1e-6,            
        verbose=1
    )
]

## 8. Compile the model 

In [ ]:
model.compile(
    optimizer='adam',             
    loss='categorical_crossentropy',
    metrics=['accuracy']           
)


## 9. Train the model

In [ ]:
history = model.fit(
    train_dataset,
    epochs=50,
    validation_data=val_dataset,
    callbacks=callbacks
)


## 10

In [ ]:
test_loss, test_acc = model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.4f}")

predictions = model.predict(test_dataset)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test_one_hot, axis=1)

In [ ]:
def plot_training_history(history):
    """Plot training and validation accuracy and loss"""
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_confusion_matrix(true_classes, predicted_classes, class_names):
    """Create and plot confusion matrix"""
    from sklearn.metrics import confusion_matrix
    
    cm = confusion_matrix(true_classes, predicted_classes)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()
    
    # Print classification report
    from sklearn.metrics import classification_report
    print("Classification Report:")
    print(classification_report(true_classes, predicted_classes, target_names=class_names))

In [ ]:
def visualize_predictions(X, y_true, y_pred, class_names, num_samples=10):
    """Visualize some example predictions"""
    indices = np.random.choice(range(len(X)), num_samples, replace=False)
    plt.figure(figsize=(15, 10))
    
    for i, idx in enumerate(indices):
        plt.subplot(2, 5, i+1)
        plt.imshow(X[idx])
        plt.title(f"True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}")
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()


In [ ]:
plot_training_history(history)
plot_confusion_matrix(true_classes, predicted_classes, class_names)

X_test_sample = X_test[:1000]  
y_true_sample = np.argmax(y_test_one_hot[:1000], axis=1)
y_pred_sample = np.argmax(model.predict(X_test_sample), axis=1)
visualize_predictions(X_test_sample, y_true_sample, y_pred_sample, class_names)

In [ ]:
model.save('best_model.keras')
